# SignBridge AI — ASL Fingerspelling-to-Text Recognition

**SignBridge AI** is a deep-learning accessibility project for recognizing continuous
American Sign Language (ASL) fingerspelling from video and converting it into text.

### Pipeline
FSBoard video → MediaPipe hand landmarks → Temporal Conv1D CNN → Bidirectional GRU → CTC → Text

Each video frame is represented by 21 hand landmarks with `(x, y, z)` coordinates,
giving **63 features per frame**. The model uses temporal convolution for local motion
patterns, BiGRU layers for longer sequence context, and Connectionist Temporal
Classification (CTC) for sequence learning without frame-level character alignment.

> **Scope:** This project recognizes ASL fingerspelling. It is not a full ASL
> language-translation system.

### Dataset
The project uses the FSBoard fingerspelling dataset. Dataset files are not included
in this repository. Follow the dataset's official access and licensing requirements.

### Security / credentials
Kaggle credentials must be supplied through Google Colab Secrets or environment
variables. Never commit API tokens or other credentials to this repository.

### Notebook
This notebook contains the project's working preprocessing, training, evaluation,
and interactive SignBridge Live demonstration pipeline.


In [ ]:
'''
                  Google FSBoard >1 TB
                         │
                         ▼
               Selective Kaggle streaming
                         │
                 10 temporary videos
                         │
                         ▼
                MediaPipe extraction
                         │
               Frames × 63 coordinates
                         │
                         ▼
              compressed float16 .npz
                         │
                  DELETE VIDEO
                         │
                  repeat 5k–15k
                         │
                         ▼
                 ┌─────────────┐
                 │ Conv1D 128  │
                 └──────┬──────┘
                        ▼
                 ┌─────────────┐
                 │ Conv1D 256  │
                 └──────┬──────┘
                        ▼
                 ┌─────────────┐
                 │ Conv1D 256  │
                 └──────┬──────┘
                        ▼
                 ┌─────────────┐
                 │ BiGRU 256×2 │
                 └──────┬──────┘
                        ▼
                       CTC
                        │
                        ▼
              Fingerspelling → Text
                        │
                        ▼
                   Live Webcam
'''

'''
“The original FSBoard dataset exceeds 1 TB. A storage-efficient streaming
preprocessing pipeline was developed to download samples incrementally, extract
normalized MediaPipe hand landmarks, persist only the compact landmark
representation, and immediately discard raw video data. This enables large-scale
CNN-based fingerspelling recognition on consumer hardware without storing the
complete raw dataset.”
'''

In [ ]:
# '''IMPORTANT CODE TO CHECK DATA. DONT RUN EVERYTIME.ONLY REQUIRED TO CHECK FILES WHEN NEEDED'''

# import json
# from pathlib import Path

# META_DIR = Path(
#     "/content/drive/MyDrive/SignBridge_FSBoard/metadata"
# )

# files = list(META_DIR.rglob("daun_v3-*.json"))

# print("Metadata files found:", len(files))

# def inspect(obj, depth=0, max_depth=3):
#     indent = "  " * depth

#     if depth > max_depth:
#         return

#     if isinstance(obj, dict):
#         print(indent + f"DICT with {len(obj)} keys")
#         print(indent + "Keys:", list(obj.keys())[:30])

#         for key in list(obj.keys())[:3]:
#             print(indent + f"\n[{key!r}] ->")
#             inspect(obj[key], depth + 1, max_depth)

#     elif isinstance(obj, list):
#         print(indent + f"LIST length: {len(obj)}")

#         if obj:
#             print(indent + "First item:")
#             inspect(obj[0], depth + 1, max_depth)

#     else:
#         print(
#             indent +
#             f"{type(obj).__name__}: "
#             f"{repr(obj)[:500]}"
#         )


# for path in files:

#     print("\n" + "=" * 80)
#     print(path)
#     print("=" * 80)

#     with open(path, "r", encoding="utf-8") as f:
#         data = json.load(f)

#     inspect(data)

In [ ]:
# import json
# from pathlib import Path
# import os

# ROOT = Path("/content/drive/MyDrive/SignBridge_FSBoard")
# META_DIR = ROOT / "metadata"
# STATE_DIR = ROOT / "state"

# # -------------------------------------------------------
# # Load previously saved FSBoard file index
# # -------------------------------------------------------

# INDEX_FILE = STATE_DIR / "fsboard_file_index.json"

# with open(INDEX_FILE, "r") as f:
#     index_data = json.load(f)

# if isinstance(index_data, dict):
#     FILE_RECORDS = index_data["records"]
# else:
#     FILE_RECORDS = index_data

# VIDEO_EXTENSIONS = (
#     ".mp4",
#     ".mov",
#     ".m4v",
#     ".webm"
# )

# VIDEO_FILES = [
#     record["name"]
#     for record in FILE_RECORDS
#     if record["name"].lower().endswith(VIDEO_EXTENSIONS)
# ]

# print("Indexed videos:", len(VIDEO_FILES))


# # -------------------------------------------------------
# # Read REAL FSBoard metadata schema
# # -------------------------------------------------------

# METADATA_FILES = list(
#     META_DIR.rglob("daun_v3-*.json")
# )

# print("Metadata files:", len(METADATA_FILES))

# LABEL_LOOKUP = {}
# SIGNER_LOOKUP = {}
# SPLIT_LOOKUP = {}

# metadata_records = 0


# for metadata_file in METADATA_FILES:

#     filename = metadata_file.name.lower()

#     if "train" in filename:
#         split = "train"

#     elif "validation" in filename:
#         split = "validation"

#     elif "test" in filename:
#         split = "test"

#     else:
#         split = "unknown"


#     print("\nReading:", metadata_file.name)


#     with open(
#         metadata_file,
#         "r",
#         encoding="utf-8"
#     ) as f:

#         records = json.load(f)


#     print("Records:", len(records))


#     for record in records:

#         # FSBoard fingerspelling only
#         if record.get("signingType") != "fs":
#             continue

#         clip_filename = record.get(
#             "clipFilename"
#         )

#         phrase = record.get(
#             "phrase"
#         )

#         signer_id = record.get(
#             "signerId",
#             "unknown"
#         )


#         if not clip_filename:
#             continue

#         if phrase is None:
#             continue


#         clip_filename = os.path.basename(
#             str(clip_filename)
#         )

#         phrase = str(
#             phrase
#         ).strip()


#         if not phrase:
#             continue


#         LABEL_LOOKUP[
#             clip_filename
#         ] = phrase

#         SIGNER_LOOKUP[
#             clip_filename
#         ] = str(
#             signer_id
#         )

#         SPLIT_LOOKUP[
#             clip_filename
#         ] = split

#         metadata_records += 1


# print("\n" + "=" * 70)

# print(
#     "Usable metadata labels:",
#     len(LABEL_LOOKUP)
# )


# # -------------------------------------------------------
# # Match indexed Kaggle paths to metadata
# # -------------------------------------------------------

# LABELED_VIDEOS = []


# for remote_video in VIDEO_FILES:

#     basename = os.path.basename(
#         remote_video
#     )


#     if basename not in LABEL_LOOKUP:
#         continue


#     LABELED_VIDEOS.append(
#         {
#             "remote_path":
#                 remote_video,

#             "filename":
#                 basename,

#             "phrase":
#                 LABEL_LOOKUP[
#                     basename
#                 ],

#             "signer":
#                 SIGNER_LOOKUP.get(
#                     basename,
#                     "unknown"
#                 ),

#             "split":
#                 SPLIT_LOOKUP.get(
#                     basename,
#                     "unknown"
#                 )
#         }
#     )


# print(
#     "Indexed videos:",
#     len(VIDEO_FILES)
# )

# print(
#     "Matched labeled videos:",
#     len(LABELED_VIDEOS)
# )

# print(
#     "Match rate:",
#     f"{100 * len(LABELED_VIDEOS) / max(1, len(VIDEO_FILES)):.2f}%"
# )


# # -------------------------------------------------------
# # Split distribution
# # -------------------------------------------------------

# split_counts = {}

# for item in LABELED_VIDEOS:

#     split = item["split"]

#     split_counts[
#         split
#     ] = split_counts.get(
#         split,
#         0
#     ) + 1


# print("\nMatched split distribution:")

# for split, count in split_counts.items():

#     print(
#         f"{split:12s}: {count:,}"
#     )


# # -------------------------------------------------------
# # Show examples
# # -------------------------------------------------------

# print("\nExamples:")

# for item in LABELED_VIDEOS[:10]:

#     print(
#         "\nVideo :",
#         item["filename"]
#     )

#     print(
#         "Phrase:",
#         repr(
#             item["phrase"]
#         )
#     )

#     print(
#         "Signer:",
#         item["signer"]
#     )

#     print(
#         "Split :",
#         item["split"]
#     )


# # -------------------------------------------------------
# # Save corrected mapping
# # -------------------------------------------------------

# MAPPING_FILE = (
#     STATE_DIR /
#     "fsboard_corrected_mapping.json"
# )


# with open(
#     MAPPING_FILE,
#     "w"
# ) as f:

#     json.dump(
#         LABELED_VIDEOS,
#         f
#     )


# print(
#     "\n✅ Corrected mapping saved:"
# )

# print(
#     MAPPING_FILE
# )

In [ ]:
# import json
# from pathlib import Path

# INDEX_FILE = Path(
#     "/content/drive/MyDrive/SignBridge_FSBoard/state/fsboard_file_index.json"
# )

# with open(INDEX_FILE, "r") as f:
#     data = json.load(f)

# records = data["records"] if isinstance(data, dict) else data

# videos = [
#     x["name"]
#     for x in records
#     if x["name"].lower().endswith(".mp4")
# ]

# print("First 10 remote Kaggle video paths:\n")

# for x in videos[:10]:
#     print(x)

In [ ]:
# =============================================================================
# 🤟 SIGNBRIDGE AI — FINAL SINGLE-CELL GOOGLE COLAB
# =============================================================================
#
# REAL-TIME ASL FINGERSPELLING → TEXT
#
# Google AI FSBoard
#       ↓
# Official Train / Validation / Test metadata
#       ↓
# Download ONE video at a time
#       ↓
# MediaPipe Tasks HandLandmarker
#       ↓
# 21 landmarks × XYZ = 63 features/frame
#       ↓
# Delete raw video immediately
#       ↓
# Save compressed landmarks to Google Drive
#       ↓
# Temporal Conv1D CNN
#       ↓
# BiGRU
#       ↓
# CTC
#       ↓
# Text
#       ↓
# Browser Webcam Demo
#
# ---------------------------------------------------------------------------
# BEFORE RUNNING
# ---------------------------------------------------------------------------
#
# 1. Colab → Runtime → Change runtime type → T4 GPU
#
# 2. Colab → 🔑 Secrets
#    Add:
#
#       KAGGLE_API_TOKEN
#
#    and enable Notebook access.
#
# ---------------------------------------------------------------------------
# FINAL PROJECT TARGET
# ---------------------------------------------------------------------------
#
# Train      : 4000
# Validation : 700
# Test       : 300
# TOTAL      : 5000
#
# FIRST RUN:
# Train      : +400
# Validation : +70
# Test       : +30
#
# Re-run THIS SAME CELL to continue growing the dataset.
#
# =============================================================================



# =============================================================================
# 0. INSTALL
# =============================================================================

!pip -q install -U \
    "pandas==2.2.3" \
    kagglehub \
    kaggle \
    mediapipe \
    gradio \
    opencv-python-headless \
    tqdm



# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import sys
import gc
import re
import json
import time
import random
import shutil
import hashlib
import warnings
import urllib.request

from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import mediapipe as mp
import kagglehub
import gradio as gr

from tqdm.auto import tqdm
from tensorflow.keras import layers, Model

warnings.filterwarnings("ignore")



# =============================================================================
# 2. ENVIRONMENT INFORMATION
# =============================================================================

print("=" * 80)
print("🤟 SIGNBRIDGE AI — FSBOARD")
print("=" * 80)

print("Python      :", sys.version.split()[0])
print("TensorFlow  :", tf.__version__)
print("Pandas      :", pd.__version__)
print("MediaPipe   :", getattr(mp, "__version__", "unknown"))

GPU_DEVICES = tf.config.list_physical_devices("GPU")

print("GPU devices :", GPU_DEVICES)

if GPU_DEVICES:
    print("✅ GPU detected.")
else:
    print("⚠️ No GPU detected. Enable a T4 GPU in Colab.")



# =============================================================================
# 3. CONFIGURATION
# =============================================================================

DATASET = "googleai/fsboard"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# -----------------------------------------------------------------------------
# FINAL STORED DATASET TARGET
# -----------------------------------------------------------------------------

FINAL_TARGETS = {
    "train": 4000,
    "validation": 700,
    "test": 300
}


# -----------------------------------------------------------------------------
# NEW VALID SAMPLES TO ADD EACH TIME THIS CELL RUNS
#
# Keep this initially.
#
# Once everything works well, you may increase these later.
# -----------------------------------------------------------------------------

RUN_TARGETS = {
    "train": 400,
    "validation": 70,
    "test": 30
}


# We choose more candidates than required because some videos may fail
# hand detection or CTC length validation.

POOL_MULTIPLIER = 2


# -----------------------------------------------------------------------------
# VIDEO SETTINGS
# -----------------------------------------------------------------------------

TARGET_FPS = 12

MAX_FRAMES = 240

MIN_FRAMES = 6

NUM_LANDMARKS = 21

FEATURES = 63


# -----------------------------------------------------------------------------
# MODEL
# -----------------------------------------------------------------------------

BATCH_SIZE = 16

EPOCHS = 15

LEARNING_RATE = 1e-3

CNN_FILTERS = 128

RNN_UNITS = 256

EARLY_STOPPING_PATIENCE = 4


# -----------------------------------------------------------------------------
# LIVE WEBCAM
# -----------------------------------------------------------------------------

LIVE_WINDOW = 100

MIN_LIVE_FRAMES = 12

LAUNCH_WEBCAM = True



# =============================================================================
# 4. GOOGLE DRIVE
# =============================================================================

print("\n" + "=" * 80)
print("GOOGLE DRIVE")
print("=" * 80)

USE_DRIVE = False

try:

    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

    USE_DRIVE = True

    print("✅ Google Drive mounted.")

except Exception as e:

    print("⚠️ Google Drive unavailable:", e)


if USE_DRIVE:

    ROOT = Path(
        "/content/drive/MyDrive/SignBridge_FSBoard"
    )

else:

    ROOT = Path(
        "/content/SignBridge_FSBoard"
    )


# V3 deliberately separates these files from earlier experimental versions.

FEATURE_DIR = ROOT / "landmarks_v3"

MODEL_DIR = ROOT / "model_v3"

STATE_DIR = ROOT / "state_v3"

META_DIR = ROOT / "metadata"

TEMP_DIR = Path(
    "/content/signbridge_temp"
)

KAGGLE_CACHE = Path(
    "/content/signbridge_kaggle_cache"
)


for directory in [
    FEATURE_DIR,
    MODEL_DIR,
    STATE_DIR,
    META_DIR,
    TEMP_DIR,
    KAGGLE_CACHE
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


os.environ["KAGGLEHUB_CACHE"] = str(
    KAGGLE_CACHE
)


print("Project folder:")
print(ROOT)



# =============================================================================
# 5. KAGGLE AUTHENTICATION
# =============================================================================

print("\n" + "=" * 80)
print("KAGGLE AUTHENTICATION")
print("=" * 80)

KAGGLE_TOKEN = None


try:

    from google.colab import userdata

    KAGGLE_TOKEN = userdata.get(
        "KAGGLE_API_TOKEN"
    )

except Exception:
    pass


if not KAGGLE_TOKEN:

    KAGGLE_TOKEN = os.environ.get(
        "KAGGLE_API_TOKEN"
    )


if not KAGGLE_TOKEN:

    raise RuntimeError(
        """
❌ KAGGLE_API_TOKEN was not found.

In Google Colab:

1. Open 🔑 Secrets
2. Create:

   KAGGLE_API_TOKEN

3. Paste your Kaggle API token.
4. Enable notebook access.
5. Run this cell again.
"""
    )


os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

print("✅ Kaggle token found.")
print("✅ Kaggle authentication configured.")



# =============================================================================
# 6. OFFICIAL FSBOARD METADATA
# =============================================================================

print("\n" + "=" * 80)
print("FSBOARD METADATA")
print("=" * 80)


METADATA_REMOTE_PATHS = {

    "train":
        "daun_v3/metadata/daun_v3-train.json",

    "validation":
        "daun_v3/metadata/daun_v3-validation.json",

    "test":
        "daun_v3/metadata/daun_v3-test.json"
}



def find_metadata_file(split):

    expected = f"daun_v3-{split}.json"

    matches = list(
        META_DIR.rglob(
            expected
        )
    )

    if matches:
        return matches[0]

    return None



METADATA_PATHS = {}


for split, remote_path in METADATA_REMOTE_PATHS.items():

    local_path = find_metadata_file(
        split
    )


    if local_path is None:

        print(
            f"Downloading {split} metadata..."
        )

        downloaded = kagglehub.dataset_download(

            DATASET,

            path=remote_path,

            output_dir=str(
                META_DIR
            )

        )

        local_path = Path(
            downloaded
        )


    METADATA_PATHS[
        split
    ] = local_path


    print(
        f"✅ {split:10s}: {local_path}"
    )



# =============================================================================
# 7. READ OFFICIAL TRAIN / VALIDATION / TEST SPLITS
# =============================================================================

print("\n" + "=" * 80)
print("READING OFFICIAL FSBOARD SPLITS")
print("=" * 80)


ALL_RECORDS = {}


for split, metadata_path in METADATA_PATHS.items():

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        raw_records = json.load(
            f
        )


    records = []


    for item in raw_records:

        # FS = fingerspelling

        if item.get(
            "signingType"
        ) != "fs":

            continue


        clip_filename = item.get(
            "clipFilename"
        )

        phrase = item.get(
            "phrase"
        )

        signer = item.get(
            "signerId",
            "unknown"
        )


        if not clip_filename:
            continue

        if phrase is None:
            continue


        # -------------------------------------------------------------
        # Confirmed FSBoard Kaggle structure:
        #
        # daun_v3/video_clips/daun_v3-train/FILE.mp4
        # daun_v3/video_clips/daun_v3-validation/FILE.mp4
        # daun_v3/video_clips/daun_v3-test/FILE.mp4
        # -------------------------------------------------------------

        remote_video_path = (

            f"daun_v3/video_clips/"
            f"daun_v3-{split}/"
            f"{clip_filename}"

        )


        records.append(

            {

                "split":
                    split,

                "filename":
                    str(
                        clip_filename
                    ),

                "remote_path":
                    remote_video_path,

                "phrase":
                    str(
                        phrase
                    ),

                "signer":
                    str(
                        signer
                    )

            }

        )


    ALL_RECORDS[
        split
    ] = records


    print(
        f"{split:10s}: {len(records):,} clips"
    )


print()
print("✅ Full Kaggle dataset indexing is NOT required.")



# =============================================================================
# 8. DETERMINISTIC PROJECT SUBSET
# =============================================================================

print("\n" + "=" * 80)
print("SELECTING PROJECT SUBSET")
print("=" * 80)


SELECTED_RECORDS = {}


SPLIT_SEEDS = {

    "train":
        1001,

    "validation":
        2002,

    "test":
        3003

}


for split in [
    "train",
    "validation",
    "test"
]:

    records = list(
        ALL_RECORDS[
            split
        ]
    )


    rng = random.Random(

        SEED
        +
        SPLIT_SEEDS[
            split
        ]

    )


    rng.shuffle(
        records
    )


    required_pool = (

        FINAL_TARGETS[
            split
        ]

        *

        POOL_MULTIPLIER

    )


    pool_size = min(

        len(
            records
        ),

        required_pool

    )


    SELECTED_RECORDS[
        split
    ] = records[
        :pool_size
    ]


    print(

        f"{split:10s}: "
        f"target {FINAL_TARGETS[split]:,} "
        f"from {pool_size:,} candidates"

    )



# =============================================================================
# 9. CHARACTER VOCABULARY
# =============================================================================

print("\n" + "=" * 80)
print("VOCABULARY")
print("=" * 80)



def normalize_phrase(text):

    text = str(
        text
    )

    text = text.replace(
        "\n",
        " "
    )

    text = text.replace(
        "\r",
        " "
    )

    text = text.replace(
        "\t",
        " "
    )


    text = re.sub(

        r"\s+",

        " ",

        text

    )


    # FSBoard metadata is essentially printable character text.
    # We retain printable ASCII characters.

    text = "".join(

        character

        for character
        in text

        if 32 <= ord(
            character
        ) <= 126

    )


    return text.strip()



ALL_PROJECT_PHRASES = []


for split in SELECTED_RECORDS:

    for record in SELECTED_RECORDS[
        split
    ]:

        phrase = normalize_phrase(
            record[
                "phrase"
            ]
        )

        if phrase:

            ALL_PROJECT_PHRASES.append(
                phrase
            )



CHARACTERS = "".join(

    sorted(

        set(

            "".join(
                ALL_PROJECT_PHRASES
            )

        ),

        key=ord

    )

)


# CTC blank = 0
# actual characters begin at 1.

CHAR_TO_ID = {

    character:
        index + 1

    for index, character
    in enumerate(
        CHARACTERS
    )

}


ID_TO_CHAR = {

    index + 1:
        character

    for index, character
    in enumerate(
        CHARACTERS
    )

}


NUM_CLASSES = (
    len(
        CHARACTERS
    )
    +
    1
)


print(
    "Characters:",
    repr(
        CHARACTERS
    )
)

print(
    "Character classes:",
    len(
        CHARACTERS
    )
)

print(
    "Output classes including CTC blank:",
    NUM_CLASSES
)



VOCAB_PATH = (
    STATE_DIR /
    "vocabulary.json"
)


with open(
    VOCAB_PATH,
    "w"
) as f:

    json.dump(

        {
            "characters":
                CHARACTERS,

            "num_classes":
                NUM_CLASSES
        },

        f,

        indent=2

    )



def encode_text(text):

    text = normalize_phrase(
        text
    )

    encoded = [

        CHAR_TO_ID[
            character
        ]

        for character
        in text

        if character in CHAR_TO_ID

    ]


    return np.asarray(

        encoded,

        dtype=np.int32

    )



def decode_ids(ids):

    output = []


    for value in ids:

        value = int(
            value
        )


        if value <= 0:
            continue


        character = ID_TO_CHAR.get(
            value
        )


        if character is not None:

            output.append(
                character
            )


    return "".join(
        output
    )



# =============================================================================
# 10. CURRENT MEDIAPIPE TASKS API
# =============================================================================

print("\n" + "=" * 80)
print("MEDIAPIPE HAND LANDMARKER")
print("=" * 80)


HAND_MODEL_PATH = Path(
    "/content/hand_landmarker.task"
)


HAND_MODEL_URL = (

    "https://storage.googleapis.com/"
    "mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/"
    "hand_landmarker.task"

)


if not HAND_MODEL_PATH.exists():

    print(
        "Downloading Hand Landmarker model..."
    )

    urllib.request.urlretrieve(

        HAND_MODEL_URL,

        HAND_MODEL_PATH

    )


print(
    "✅ MediaPipe Hand Landmarker model ready."
)


# Verify modern API exists.

if not hasattr(
    mp,
    "tasks"
):

    raise RuntimeError(
        """
The installed MediaPipe build does not expose mp.tasks.

Restart the Colab runtime once and run this SAME cell again.
"""
    )


BaseOptions = mp.tasks.BaseOptions

HandLandmarker = (
    mp.tasks.vision.HandLandmarker
)

HandLandmarkerOptions = (
    mp.tasks.vision.HandLandmarkerOptions
)

VisionRunningMode = (
    mp.tasks.vision.RunningMode
)



# =============================================================================
# 11. LANDMARK NORMALIZATION
# =============================================================================

def normalize_landmarks(xyz):

    xyz = np.asarray(

        xyz,

        dtype=np.float32

    )


    # Wrist becomes origin.

    xyz = xyz - xyz[0]


    # Scale normalization:
    # wrist → middle finger MCP = landmark 9

    scale = np.linalg.norm(
        xyz[
            9
        ]
    )


    if scale < 1e-6:

        scale = 1.0


    xyz = xyz / scale


    return xyz



# =============================================================================
# 12. VIDEO → LANDMARK SEQUENCE
# =============================================================================

def extract_landmarks(video_path):

    capture = cv2.VideoCapture(

        str(
            video_path
        )

    )


    if not capture.isOpened():

        return None


    source_fps = capture.get(
        cv2.CAP_PROP_FPS
    )


    if (

        not source_fps

        or

        np.isnan(
            source_fps
        )

        or

        source_fps <= 0

    ):

        source_fps = 30.0


    frame_step = max(

        1,

        int(

            round(

                source_fps
                /
                TARGET_FPS

            )

        )

    )


    options = HandLandmarkerOptions(

        base_options=BaseOptions(

            model_asset_path=str(
                HAND_MODEL_PATH
            )

        ),

        running_mode=VisionRunningMode.VIDEO,

        num_hands=1,

        min_hand_detection_confidence=0.35,

        min_hand_presence_confidence=0.35,

        min_tracking_confidence=0.35

    )


    sequence = []

    frame_number = 0

    sampled_index = 0


    with HandLandmarker.create_from_options(
        options
    ) as hand_landmarker:


        while True:

            success, frame = capture.read()


            if not success:
                break


            should_process = (

                frame_number
                %
                frame_step
                ==
                0

            )


            frame_number += 1


            if not should_process:
                continue


            rgb = cv2.cvtColor(

                frame,

                cv2.COLOR_BGR2RGB

            )


            # Ensure contiguous memory.

            rgb = np.ascontiguousarray(
                rgb
            )


            mediapipe_image = mp.Image(

                image_format=mp.ImageFormat.SRGB,

                data=rgb

            )


            # VIDEO mode timestamps must continuously increase.

            timestamp_ms = int(

                sampled_index
                *
                (
                    1000.0
                    /
                    TARGET_FPS
                )

            )


            sampled_index += 1


            result = hand_landmarker.detect_for_video(

                mediapipe_image,

                timestamp_ms

            )


            if result.hand_landmarks:

                hand = result.hand_landmarks[
                    0
                ]


                xyz = np.asarray(

                    [

                        [

                            landmark.x,
                            landmark.y,
                            landmark.z

                        ]

                        for landmark
                        in hand

                    ],

                    dtype=np.float32

                )


                if xyz.shape == (
                    NUM_LANDMARKS,
                    3
                ):

                    xyz = normalize_landmarks(
                        xyz
                    )


                    sequence.append(

                        xyz.reshape(
                            FEATURES
                        )

                    )


            elif sequence:

                # Keep temporal continuity through occasional
                # single-frame detector dropout.

                sequence.append(

                    sequence[
                        -1
                    ].copy()

                )


            if len(
                sequence
            ) >= MAX_FRAMES:

                break


    capture.release()


    if len(
        sequence
    ) < MIN_FRAMES:

        return None


    return np.asarray(

        sequence,

        dtype=np.float32

    )



print(
    "✅ MediaPipe Tasks preprocessing ready."
)



# =============================================================================
# 13. CTC MINIMUM TIMESTEPS
# =============================================================================

def required_ctc_frames(encoded):

    encoded = list(
        encoded
    )


    if not encoded:

        return 0


    duplicate_neighbours = 0


    for index in range(

        1,

        len(
            encoded
        )

    ):

        if (

            encoded[
                index
            ]

            ==

            encoded[
                index - 1
            ]

        ):

            duplicate_neighbours += 1


    return (

        len(
            encoded
        )

        +

        duplicate_neighbours

    )



# =============================================================================
# 14. MANIFEST
# =============================================================================

MANIFEST_PATH = (
    STATE_DIR /
    "manifest.json"
)


if MANIFEST_PATH.exists():

    try:

        with open(
            MANIFEST_PATH,
            "r"
        ) as f:

            MANIFEST = json.load(
                f
            )

    except Exception:

        MANIFEST = {
            "processed": {}
        }

else:

    MANIFEST = {
        "processed": {}
    }



def save_manifest():

    temporary_path = (
        STATE_DIR /
        "manifest.tmp"
    )


    with open(
        temporary_path,
        "w"
    ) as f:

        json.dump(
            MANIFEST,
            f
        )


    os.replace(

        temporary_path,

        MANIFEST_PATH

    )



def feature_path_for(remote_path):

    digest = hashlib.sha1(

        remote_path.encode(
            "utf-8"
        )

    ).hexdigest()


    return (

        FEATURE_DIR /
        f"{digest}.npz"

    )



# =============================================================================
# 15. EXISTING VALID SAMPLE COUNT
# =============================================================================

def existing_valid_for_split(split):

    count = 0


    for source, info in MANIFEST[
        "processed"
    ].items():

        if info.get(
            "status"
        ) != "ok":

            continue


        if info.get(
            "split"
        ) != split:

            continue


        filename = info.get(
            "feature_file"
        )


        if not filename:
            continue


        if (
            FEATURE_DIR /
            filename
        ).exists():

            count += 1


    return count



print("\n" + "=" * 80)
print("CACHE STATUS")
print("=" * 80)


for split in [
    "train",
    "validation",
    "test"
]:

    print(

        f"{split:10s}: "
        f"{existing_valid_for_split(split):,} / "
        f"{FINAL_TARGETS[split]:,}"

    )



# =============================================================================
# 16. SAFE INDIVIDUAL KAGGLE DOWNLOAD
# =============================================================================

def download_single_video(
    remote_path,
    max_attempts=5
):

    last_error = None


    for attempt in range(
        max_attempts
    ):

        try:

            downloaded = (
                kagglehub.dataset_download(

                    DATASET,

                    path=remote_path,

                    output_dir=str(
                        TEMP_DIR
                    )

                )
            )


            result = Path(
                downloaded
            )


            if result.exists():

                return result


            raise FileNotFoundError(
                f"Downloaded file was not found: {result}"
            )


        except Exception as e:

            last_error = e


            error_text = str(
                e
            )


            if "429" in error_text:

                delay = min(

                    10
                    *
                    (
                        2 ** attempt
                    ),

                    90

                )


                print(
                    f"\n⚠️ Kaggle rate limit. "
                    f"Retrying in {delay}s..."
                )


                time.sleep(
                    delay
                )


            else:

                if attempt < (
                    max_attempts - 1
                ):

                    time.sleep(
                        3
                    )


    raise last_error



# =============================================================================
# 17. STREAM FSBOARD → LANDMARKS
# =============================================================================

print("\n" + "=" * 80)
print("STREAMING FSBOARD")
print("=" * 80)

print(
    "Videos are downloaded one-at-a-time and deleted after preprocessing."
)


for split in [
    "train",
    "validation",
    "test"
]:

    existing = existing_valid_for_split(
        split
    )


    remaining_final = max(

        0,

        FINAL_TARGETS[
            split
        ]
        -
        existing

    )


    target_this_run = min(

        RUN_TARGETS[
            split
        ],

        remaining_final

    )


    print()
    print("-" * 75)

    print(

        f"{split.upper()} | "
        f"existing={existing:,} | "
        f"new target={target_this_run:,}"

    )


    if target_this_run <= 0:

        print(
            "✅ Target already reached."
        )

        continue


    new_valid = 0

    attempted = 0


    for record in SELECTED_RECORDS[
        split
    ]:

        if new_valid >= target_this_run:
            break


        remote_path = record[
            "remote_path"
        ]


        previous = MANIFEST[
            "processed"
        ].get(
            remote_path
        )


        if previous:

            previous_status = previous.get(
                "status"
            )


            if previous_status in {

                "ok",
                "no_hand",
                "ctc_too_short",
                "empty_label"

            }:

                continue


        attempted += 1

        local_video = None


        try:

            # ---------------------------------------------------------
            # Clean local temporary directory
            # ---------------------------------------------------------

            shutil.rmtree(

                TEMP_DIR,

                ignore_errors=True

            )


            TEMP_DIR.mkdir(

                parents=True,

                exist_ok=True

            )


            # ---------------------------------------------------------
            # Download ONE raw FSBoard video
            # ---------------------------------------------------------

            local_video = download_single_video(
                remote_path
            )


            # ---------------------------------------------------------
            # Video → MediaPipe hand landmarks
            # ---------------------------------------------------------

            sequence = extract_landmarks(
                local_video
            )


            if sequence is None:

                MANIFEST[
                    "processed"
                ][remote_path] = {

                    "status":
                        "no_hand",

                    "split":
                        split

                }


                save_manifest()

                continue


            # ---------------------------------------------------------
            # Ground truth
            # ---------------------------------------------------------

            phrase = normalize_phrase(
                record[
                    "phrase"
                ]
            )


            encoded = encode_text(
                phrase
            )


            if len(
                encoded
            ) == 0:

                MANIFEST[
                    "processed"
                ][remote_path] = {

                    "status":
                        "empty_label",

                    "split":
                        split

                }


                save_manifest()

                continue


            # ---------------------------------------------------------
            # CTC compatibility
            # ---------------------------------------------------------

            if (

                len(
                    sequence
                )

                <

                required_ctc_frames(
                    encoded
                )

            ):

                MANIFEST[
                    "processed"
                ][remote_path] = {

                    "status":
                        "ctc_too_short",

                    "split":
                        split

                }


                save_manifest()

                continue


            # ---------------------------------------------------------
            # Save compact landmark representation
            # ---------------------------------------------------------

            destination = (
                feature_path_for(
                    remote_path
                )
            )


            np.savez_compressed(

                destination,

                x=
                    sequence.astype(
                        np.float16
                    ),

                y=
                    encoded.astype(
                        np.int16
                    ),

                text=
                    np.asarray(
                        phrase
                    ),

                signer=
                    np.asarray(
                        record[
                            "signer"
                        ]
                    ),

                split=
                    np.asarray(
                        split
                    ),

                source=
                    np.asarray(
                        remote_path
                    )

            )


            MANIFEST[
                "processed"
            ][remote_path] = {

                "status":
                    "ok",

                "split":
                    split,

                "feature_file":
                    destination.name,

                "signer":
                    record[
                        "signer"
                    ],

                "phrase":
                    phrase,

                "frames":
                    int(
                        len(
                            sequence
                        )
                    )

            }


            save_manifest()


            new_valid += 1


            current = (
                existing
                +
                new_valid
            )


            print(

                f"\r✅ {split:10s} "
                f"{current:,}/"
                f"{FINAL_TARGETS[split]:,} "
                f"| this run "
                f"{new_valid:,}/"
                f"{target_this_run:,} "
                f"| attempts {attempted:,}",

                end=""

            )


        except Exception as e:

            MANIFEST[
                "processed"
            ][remote_path] = {

                "status":
                    "error",

                "split":
                    split,

                "error":
                    str(
                        e
                    )[:400]

            }


            save_manifest()


        finally:

            # ---------------------------------------------------------
            # Delete downloaded raw video.
            # ---------------------------------------------------------

            try:

                if (

                    local_video is not None

                    and

                    local_video.exists()

                ):

                    local_video.unlink()

            except Exception:
                pass


            shutil.rmtree(

                TEMP_DIR,

                ignore_errors=True

            )


            # Prevent Kaggle cache growth.

            shutil.rmtree(

                KAGGLE_CACHE,

                ignore_errors=True

            )


            KAGGLE_CACHE.mkdir(

                parents=True,

                exist_ok=True

            )


            gc.collect()


    print()



# =============================================================================
# 18. BUILD FEATURE FILE LISTS
# =============================================================================

FEATURE_FILES = {

    "train":
        [],

    "validation":
        [],

    "test":
        []

}


for remote_path, info in MANIFEST[
    "processed"
].items():

    if info.get(
        "status"
    ) != "ok":

        continue


    split = info.get(
        "split"
    )


    if split not in FEATURE_FILES:
        continue


    filename = info.get(
        "feature_file"
    )


    if not filename:
        continue


    path = (
        FEATURE_DIR /
        filename
    )


    if path.exists():

        FEATURE_FILES[
            split
        ].append(
            path
        )


for split in FEATURE_FILES:

    FEATURE_FILES[
        split
    ] = sorted(

        FEATURE_FILES[
            split
        ]

    )



def directory_size_mb(directory):

    total_bytes = 0


    for root, _, filenames in os.walk(
        directory
    ):

        for filename in filenames:

            path = os.path.join(
                root,
                filename
            )


            try:

                total_bytes += os.path.getsize(
                    path
                )

            except Exception:
                pass


    return (

        total_bytes
        /
        (1024 ** 2)

    )



print("\n" + "=" * 80)
print("PREPROCESSING SUMMARY")
print("=" * 80)


for split in [
    "train",
    "validation",
    "test"
]:

    print(

        f"{split:10s}: "
        f"{len(FEATURE_FILES[split]):,}"

    )


print(
    f"\nCompressed landmark storage: "
    f"{directory_size_mb(FEATURE_DIR):.2f} MB"
)

print(
    "Raw FSBoard videos retained: 0"
)



# =============================================================================
# 19. MINIMUM DATA FOR MODEL TRAINING
# =============================================================================

MIN_TRAIN_SAMPLES = 100
MIN_VAL_SAMPLES = 20
MIN_TEST_SAMPLES = 10


if (

    len(
        FEATURE_FILES[
            "train"
        ]
    )
    <
    MIN_TRAIN_SAMPLES

    or

    len(
        FEATURE_FILES[
            "validation"
        ]
    )
    <
    MIN_VAL_SAMPLES

):

    raise RuntimeError(
        """
Not enough valid samples have been cached for training yet.

Run THIS SAME CELL again.

Everything successfully processed so far is already saved in
Google Drive and will not be downloaded again.
"""
    )



# =============================================================================
# 20. TF.DATA
# =============================================================================

def feature_generator(files):

    for path in files:

        try:

            data = np.load(

                path,

                allow_pickle=False

            )


            x = data[
                "x"
            ].astype(
                np.float32
            )


            y = data[
                "y"
            ].astype(
                np.int32
            )


            if (

                len(
                    x
                )

                <

                required_ctc_frames(
                    y
                )

            ):

                continue


            yield {

                "landmarks":
                    x,

                "labels":
                    y,

                "input_length":
                    np.int32(
                        len(
                            x
                        )
                    ),

                "label_length":
                    np.int32(
                        len(
                            y
                        )
                    )

            }


        except Exception:
            continue



OUTPUT_SIGNATURE = {

    "landmarks":

        tf.TensorSpec(

            shape=(
                None,
                FEATURES
            ),

            dtype=tf.float32

        ),


    "labels":

        tf.TensorSpec(

            shape=(
                None,
            ),

            dtype=tf.int32

        ),


    "input_length":

        tf.TensorSpec(

            shape=(),

            dtype=tf.int32

        ),


    "label_length":

        tf.TensorSpec(

            shape=(),

            dtype=tf.int32

        )

}



def make_dataset(
    files,
    training=False
):

    dataset = tf.data.Dataset.from_generator(

        lambda:
            feature_generator(
                files
            ),

        output_signature=
            OUTPUT_SIGNATURE

    )


    if training:

        dataset = dataset.shuffle(

            min(
                len(
                    files
                ),
                2000
            ),

            seed=SEED,

            reshuffle_each_iteration=True

        )


    dataset = dataset.padded_batch(

        BATCH_SIZE,

        padded_shapes={

            "landmarks":
                [
                    None,
                    FEATURES
                ],

            "labels":
                [
                    None
                ],

            "input_length":
                [],

            "label_length":
                []

        },

        padding_values={

            "landmarks":
                tf.constant(
                    0,
                    tf.float32
                ),

            "labels":
                tf.constant(
                    0,
                    tf.int32
                ),

            "input_length":
                tf.constant(
                    0,
                    tf.int32
                ),

            "label_length":
                tf.constant(
                    0,
                    tf.int32
                )

        }

    )


    return dataset.prefetch(
        tf.data.AUTOTUNE
    )



TRAIN_DATASET = make_dataset(

    FEATURE_FILES[
        "train"
    ],

    training=True

)


VAL_DATASET = make_dataset(

    FEATURE_FILES[
        "validation"
    ],

    training=False

)


TEST_DATASET = make_dataset(

    FEATURE_FILES[
        "test"
    ],

    training=False

)



# =============================================================================
# 21. CNN + BiGRU MODEL
# =============================================================================

print("\n" + "=" * 80)
print("BUILDING CNN + BiGRU + CTC MODEL")
print("=" * 80)



def create_model():

    inputs = layers.Input(

        shape=(
            None,
            FEATURES
        ),

        name="landmarks"

    )


    # -----------------------------------------------------------------
    # CNN BLOCK 1
    # -----------------------------------------------------------------

    x = layers.Conv1D(

        CNN_FILTERS,

        kernel_size=5,

        padding="same",

        activation="relu",

        name="temporal_cnn_1"

    )(inputs)


    x = layers.BatchNormalization()(x)


    x = layers.SpatialDropout1D(
        0.20
    )(x)


    # -----------------------------------------------------------------
    # CNN BLOCK 2
    # -----------------------------------------------------------------

    x = layers.Conv1D(

        CNN_FILTERS * 2,

        kernel_size=3,

        padding="same",

        activation="relu",

        name="temporal_cnn_2"

    )(x)


    x = layers.BatchNormalization()(x)


    # -----------------------------------------------------------------
    # RESIDUAL CNN
    # -----------------------------------------------------------------

    residual = x


    x = layers.Conv1D(

        CNN_FILTERS * 2,

        kernel_size=3,

        padding="same",

        activation="relu",

        name="temporal_cnn_residual"

    )(x)


    x = layers.BatchNormalization()(x)


    x = layers.Add()(
        [
            residual,
            x
        ]
    )


    x = layers.Dropout(
        0.20
    )(x)


    # -----------------------------------------------------------------
    # BIDIRECTIONAL GRU
    # -----------------------------------------------------------------

    x = layers.Bidirectional(

        layers.GRU(

            RNN_UNITS,

            return_sequences=True,

            dropout=0.20

        ),

        name="bigru_1"

    )(x)


    x = layers.Bidirectional(

        layers.GRU(

            RNN_UNITS,

            return_sequences=True,

            dropout=0.20

        ),

        name="bigru_2"

    )(x)


    x = layers.Dropout(
        0.25
    )(x)


    # -----------------------------------------------------------------
    # CHARACTER LOGITS
    # -----------------------------------------------------------------

    logits = layers.Dense(

        NUM_CLASSES,

        activation=None,

        name="character_logits"

    )(x)


    return Model(

        inputs=inputs,

        outputs=logits,

        name="SignBridge_CNN_BiGRU"

    )



MODEL_PATH = (
    MODEL_DIR /
    "best_signbridge.keras"
)


if MODEL_PATH.exists():

    try:

        model = tf.keras.models.load_model(

            MODEL_PATH,

            compile=False

        )


        if (

            model.output_shape[
                -1
            ]

            !=

            NUM_CLASSES

        ):

            print(
                "⚠️ Vocabulary mismatch. Creating new model."
            )

            model = create_model()

        else:

            print(
                "✅ Existing SignBridge model loaded."
            )


    except Exception:

        model = create_model()


else:

    model = create_model()



model.summary()



# =============================================================================
# 22. OPTIMIZER + CTC LOSS
# =============================================================================

optimizer = tf.keras.optimizers.Adam(

    learning_rate=
        LEARNING_RATE,

    clipnorm=
        1.0

)



@tf.function
def calculate_ctc_loss(

    labels,
    logits,
    label_lengths,
    input_lengths

):

    losses = tf.nn.ctc_loss(

        labels=
            labels,

        logits=
            logits,

        label_length=
            label_lengths,

        logit_length=
            input_lengths,

        logits_time_major=
            False,

        blank_index=
            0

    )


    return tf.reduce_mean(
        losses
    )



# =============================================================================
# 23. CTC DECODER
# =============================================================================

def decode_predictions(
    logits,
    lengths=None
):

    logits = tf.convert_to_tensor(
        logits
    )


    batch_size = tf.shape(
        logits
    )[0]


    time_steps = tf.shape(
        logits
    )[1]


    if lengths is None:

        lengths = tf.fill(

            [
                batch_size
            ],

            time_steps

        )


    time_major_logits = tf.transpose(

        logits,

        [
            1,
            0,
            2
        ]

    )


    decoded, _ = tf.nn.ctc_greedy_decoder(

        inputs=
            time_major_logits,

        sequence_length=
            tf.cast(
                lengths,
                tf.int32
            ),

        merge_repeated=
            True,

        blank_index=
            0

    )


    dense = tf.sparse.to_dense(

        decoded[
            0
        ],

        default_value=
            0

    ).numpy()


    return [

        decode_ids(
            row
        )

        for row
        in dense

    ]



# =============================================================================
# 24. CHARACTER ERROR RATE
# =============================================================================

def levenshtein_distance(
    first,
    second
):

    previous = list(

        range(
            len(
                second
            )
            +
            1
        )

    )


    for i, first_char in enumerate(
        first,
        1
    ):

        current = [
            i
        ]


        for j, second_char in enumerate(
            second,
            1
        ):

            insertion = (
                current[
                    j - 1
                ]
                +
                1
            )


            deletion = (
                previous[
                    j
                ]
                +
                1
            )


            substitution = (

                previous[
                    j - 1
                ]

                +

                (
                    0
                    if first_char == second_char
                    else 1
                )

            )


            current.append(

                min(
                    insertion,
                    deletion,
                    substitution
                )

            )


        previous = current


    return previous[
        -1
    ]



def character_error_rate(
    truth,
    prediction
):

    if not truth:

        return 0.0


    return (

        levenshtein_distance(

            truth,
            prediction

        )

        /

        len(
            truth
        )

    )



# =============================================================================
# 25. TRAIN
# =============================================================================

print("\n" + "=" * 80)
print("TRAINING")
print("=" * 80)


best_validation_cer = float(
    "inf"
)

epochs_without_improvement = 0



for epoch in range(
    1,
    EPOCHS + 1
):

    # -----------------------------------------------------------------
    # TRAINING
    # -----------------------------------------------------------------

    train_losses = []


    train_progress = tqdm(

        TRAIN_DATASET,

        desc=
            f"Epoch {epoch}/{EPOCHS}"

    )


    for batch in train_progress:

        with tf.GradientTape() as tape:

            logits = model(

                batch[
                    "landmarks"
                ],

                training=True

            )


            loss = calculate_ctc_loss(

                batch[
                    "labels"
                ],

                logits,

                batch[
                    "label_length"
                ],

                batch[
                    "input_length"
                ]

            )


        gradients = tape.gradient(

            loss,

            model.trainable_variables

        )


        gradient_pairs = [

            (
                gradient,
                variable
            )

            for gradient, variable
            in zip(

                gradients,

                model.trainable_variables

            )

            if gradient is not None

        ]


        optimizer.apply_gradients(
            gradient_pairs
        )


        loss_value = float(
            loss.numpy()
        )


        train_losses.append(
            loss_value
        )


        train_progress.set_postfix(

            loss=
                f"{loss_value:.3f}"

        )


    # -----------------------------------------------------------------
    # VALIDATION
    # -----------------------------------------------------------------

    val_losses = []

    val_cers = []

    sample_truth = None

    sample_prediction = None


    for batch in VAL_DATASET:

        logits = model(

            batch[
                "landmarks"
            ],

            training=False

        )


        val_loss = calculate_ctc_loss(

            batch[
                "labels"
            ],

            logits,

            batch[
                "label_length"
            ],

            batch[
                "input_length"
            ]

        )


        val_losses.append(

            float(
                val_loss.numpy()
            )

        )


        predictions = decode_predictions(

            logits,

            batch[
                "input_length"
            ]

        )


        label_values = batch[
            "labels"
        ].numpy()


        label_lengths = batch[
            "label_length"
        ].numpy()


        for index, prediction in enumerate(
            predictions
        ):

            truth = decode_ids(

                label_values[
                    index
                ][

                    :
                    label_lengths[
                        index
                    ]

                ]

            )


            val_cers.append(

                character_error_rate(

                    truth,
                    prediction

                )

            )


            if sample_truth is None:

                sample_truth = truth

                sample_prediction = prediction


    mean_train_loss = float(

        np.mean(
            train_losses
        )

    )


    mean_val_loss = float(

        np.mean(
            val_losses
        )

    )


    mean_val_cer = float(

        np.mean(
            val_cers
        )

    )


    print()

    print(
        f"Epoch {epoch}/{EPOCHS}"
    )

    print(
        f"Train Loss : {mean_train_loss:.4f}"
    )

    print(
        f"Val Loss   : {mean_val_loss:.4f}"
    )

    print(
        f"Val CER    : {mean_val_cer * 100:.2f}%"
    )

    print(
        "Truth      :",
        repr(
            sample_truth
        )
    )

    print(
        "Prediction :",
        repr(
            sample_prediction
        )
    )


    # -----------------------------------------------------------------
    # BEST MODEL
    # -----------------------------------------------------------------

    if mean_val_cer < best_validation_cer:

        best_validation_cer = mean_val_cer

        epochs_without_improvement = 0


        model.save(
            MODEL_PATH
        )


        print(
            "✅ Best model saved."
        )


    else:

        epochs_without_improvement += 1


        if (

            epochs_without_improvement

            >=

            EARLY_STOPPING_PATIENCE

        ):

            print(
                "✅ Early stopping activated."
            )

            break



# =============================================================================
# 26. RESTORE BEST MODEL
# =============================================================================

if MODEL_PATH.exists():

    model = tf.keras.models.load_model(

        MODEL_PATH,

        compile=False

    )


    print(
        "✅ Best model restored."
    )



# =============================================================================
# 27. TEST SET EVALUATION
# =============================================================================

print("\n" + "=" * 80)
print("OFFICIAL FSBOARD TEST-SPLIT EVALUATION")
print("=" * 80)


TEST_CERS = []

TEST_EXAMPLES = []


if len(
    FEATURE_FILES[
        "test"
    ]
) >= MIN_TEST_SAMPLES:


    for batch in TEST_DATASET:

        logits = model(

            batch[
                "landmarks"
            ],

            training=False

        )


        predictions = decode_predictions(

            logits,

            batch[
                "input_length"
            ]

        )


        label_values = batch[
            "labels"
        ].numpy()


        label_lengths = batch[
            "label_length"
        ].numpy()


        for index, prediction in enumerate(
            predictions
        ):

            truth = decode_ids(

                label_values[
                    index
                ][

                    :
                    label_lengths[
                        index
                    ]

                ]

            )


            TEST_CERS.append(

                character_error_rate(

                    truth,
                    prediction

                )

            )


            if len(
                TEST_EXAMPLES
            ) < 10:

                TEST_EXAMPLES.append(

                    (
                        truth,
                        prediction
                    )

                )


    TEST_CER = float(

        np.mean(
            TEST_CERS
        )

    )


    print(
        f"Test CER: {TEST_CER * 100:.2f}%"
    )


    print(
        "\nExample test predictions:"
    )


    for truth, prediction in TEST_EXAMPLES:

        print()

        print(
            "Truth     :",
            repr(
                truth
            )
        )

        print(
            "Prediction:",
            repr(
                prediction
            )
        )


else:

    TEST_CER = None

    print(
        "Not enough test examples cached yet."
    )



# =============================================================================
# 28. SAVE PROJECT INFORMATION
# =============================================================================

PROJECT_INFO = {

    "project":
        "SignBridge AI",

    "dataset":
        "Google AI FSBoard",

    "task":
        "ASL fingerspelling recognition",

    "architecture":
        "MediaPipe HandLandmarker + Temporal Conv1D CNN + BiGRU + CTC",

    "train_samples":
        len(
            FEATURE_FILES[
                "train"
            ]
        ),

    "validation_samples":
        len(
            FEATURE_FILES[
                "validation"
            ]
        ),

    "test_samples":
        len(
            FEATURE_FILES[
                "test"
            ]
        ),

    "final_train_target":
        FINAL_TARGETS[
            "train"
        ],

    "final_validation_target":
        FINAL_TARGETS[
            "validation"
        ],

    "final_test_target":
        FINAL_TARGETS[
            "test"
        ],

    "target_fps":
        TARGET_FPS,

    "features_per_frame":
        FEATURES,

    "characters":
        CHARACTERS,

    "best_validation_cer":

        None

        if not np.isfinite(
            best_validation_cer
        )

        else float(
            best_validation_cer
        ),

    "test_cer":
        TEST_CER

}



with open(

    MODEL_DIR /
    "project_info.json",

    "w"

) as f:

    json.dump(

        PROJECT_INFO,

        f,

        indent=2

    )



# =============================================================================
# 29. FINAL PROJECT STATUS
# =============================================================================

print("\n" + "=" * 80)
print("🤟 SIGNBRIDGE AI — STATUS")
print("=" * 80)


print(

f"""
Dataset
-------

Google AI FSBoard

Official FSBoard splits are preserved.

Train:
{len(FEATURE_FILES['train']):,} / {FINAL_TARGETS['train']:,}

Validation:
{len(FEATURE_FILES['validation']):,} / {FINAL_TARGETS['validation']:,}

Test:
{len(FEATURE_FILES['test']):,} / {FINAL_TARGETS['test']:,}


Storage
-------

Compressed landmark features:
{directory_size_mb(FEATURE_DIR):.2f} MB

Raw videos retained:
0


Architecture
------------

Video
→ MediaPipe HandLandmarker
→ 21 × XYZ landmarks
→ 63 features/frame
→ Temporal Conv1D CNN
→ Residual CNN
→ Bidirectional GRU
→ Bidirectional GRU
→ CTC
→ Fingerspelled Text


Saved Model
-----------

{MODEL_PATH}
"""
)



FINAL_DATASET_COMPLETE = all(

    len(
        FEATURE_FILES[
            split
        ]
    )

    >=

    FINAL_TARGETS[
        split
    ]

    for split in FINAL_TARGETS

)


if FINAL_DATASET_COMPLETE:

    print(
        "🎉 Final 5,000-sample project dataset reached."
    )

else:

    print(
        "ℹ️ Final 5,000-sample target has not been reached yet."
    )

    print(
        "✅ Re-run THIS SAME CELL later to add more samples."
    )



# =============================================================================
# 30. LIVE MEDIAPIPE HAND LANDMARKER
# =============================================================================

LIVE_BUFFER = deque(

    maxlen=
        LIVE_WINDOW

)


LIVE_OPTIONS = HandLandmarkerOptions(

    base_options=BaseOptions(

        model_asset_path=str(
            HAND_MODEL_PATH
        )

    ),

    running_mode=
        VisionRunningMode.IMAGE,

    num_hands=
        1,

    min_hand_detection_confidence=
        0.50,

    min_hand_presence_confidence=
        0.50

)


LIVE_LANDMARKER = (
    HandLandmarker
    .create_from_options(
        LIVE_OPTIONS
    )
)


# Hand skeleton connections

HAND_CONNECTIONS = [

    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),

    (0, 5),
    (5, 6),
    (6, 7),
    (7, 8),

    (5, 9),
    (9, 10),
    (10, 11),
    (11, 12),

    (9, 13),
    (13, 14),
    (14, 15),
    (15, 16),

    (13, 17),
    (17, 18),
    (18, 19),
    (19, 20),

    (0, 17)

]



# =============================================================================
# 31. LIVE PREDICTION
# =============================================================================

def live_predict(frame):

    if frame is None:

        return (
            None,
            "Waiting for camera..."
        )


    image = np.asarray(
        frame
    ).copy()


    image = np.ascontiguousarray(
        image
    )


    mp_image = mp.Image(

        image_format=
            mp.ImageFormat.SRGB,

        data=
            image

    )


    result = LIVE_LANDMARKER.detect(
        mp_image
    )


    if result.hand_landmarks:

        hand = result.hand_landmarks[
            0
        ]


        xyz = np.asarray(

            [

                [

                    landmark.x,
                    landmark.y,
                    landmark.z

                ]

                for landmark in hand

            ],

            dtype=np.float32

        )


        if xyz.shape == (
            NUM_LANDMARKS,
            3
        ):

            xyz = normalize_landmarks(
                xyz
            )


            LIVE_BUFFER.append(

                xyz.reshape(
                    FEATURES
                )

            )


        # -------------------------------------------------------------
        # Draw hand landmarks
        # -------------------------------------------------------------

        height, width = image.shape[
            :2
        ]


        points = []


        for landmark in hand:

            px = int(

                landmark.x
                *
                width

            )


            py = int(

                landmark.y
                *
                height

            )


            points.append(
                (
                    px,
                    py
                )
            )


            cv2.circle(

                image,

                (
                    px,
                    py
                ),

                4,

                (
                    0,
                    255,
                    0
                ),

                -1

            )


        for start, end in HAND_CONNECTIONS:

            if (

                start < len(
                    points
                )

                and

                end < len(
                    points
                )

            ):

                cv2.line(

                    image,

                    points[
                        start
                    ],

                    points[
                        end
                    ],

                    (
                        0,
                        255,
                        0
                    ),

                    2

                )


    # -----------------------------------------------------------------
    # Temporal recognition
    # -----------------------------------------------------------------

    if len(
        LIVE_BUFFER
    ) >= MIN_LIVE_FRAMES:

        sequence = np.asarray(

            LIVE_BUFFER,

            dtype=np.float32

        )[None, ...]


        logits = model(

            sequence,

            training=False

        )


        prediction = decode_predictions(
            logits
        )[0]


        if not prediction:

            prediction = "Recognizing..."


    else:

        prediction = (

            f"Collecting movement "
            f"{len(LIVE_BUFFER)}/"
            f"{MIN_LIVE_FRAMES}"

        )


    # -----------------------------------------------------------------
    # Overlay
    # -----------------------------------------------------------------

    cv2.rectangle(

        image,

        (
            0,
            0
        ),

        (
            image.shape[
                1
            ],
            90
        ),

        (
            0,
            0,
            0
        ),

        -1

    )


    cv2.putText(

        image,

        "SignBridge AI",

        (
            15,
            30
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.70,

        (
            255,
            255,
            255
        ),

        2,

        cv2.LINE_AA

    )


    cv2.putText(

        image,

        prediction[
            :50
        ],

        (
            15,
            65
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.70,

        (
            255,
            255,
            255
        ),

        2,

        cv2.LINE_AA

    )


    return (

        image,

        prediction

    )



def clear_live_sequence():

    LIVE_BUFFER.clear()

    return (
        "Sequence cleared. Start fingerspelling."
    )



# =============================================================================
# 32. GRADIO LIVE APP
# =============================================================================

if LAUNCH_WEBCAM:

    print("\n" + "=" * 80)
    print("STARTING SIGNBRIDGE LIVE DEMO")
    print("=" * 80)


    with gr.Blocks(

        title=
            "SignBridge AI"

    ) as demo:


        gr.Markdown(
            """
# 🤟 SignBridge AI

## Real-Time ASL Fingerspelling → Text

**Google AI FSBoard → MediaPipe → CNN → BiGRU → CTC**

SignBridge analyzes a **sequence of hand movements**, rather than
classifying one isolated hand image.

This allows the system to learn temporal fingerspelling patterns,
including motion-dependent letters such as **J** and **Z**.

### Testing

1. Allow webcam access.
2. Keep your hand clearly visible.
3. Click **Clear / Start New Phrase** before a new phrase.
4. Fingerspell naturally.
5. Watch the recognized text update.
"""
        )


        with gr.Row():

            webcam = gr.Image(

                sources=[
                    "webcam"
                ],

                type=
                    "numpy",

                streaming=
                    True,

                label=
                    "Live Camera"

            )


            visual_output = gr.Image(

                label=
                    "Hand Tracking"

            )


        recognized_text = gr.Textbox(

            label=
                "Recognized Fingerspelling",

            lines=
                2

        )


        clear_button = gr.Button(

            "Clear / Start New Phrase"

        )


        webcam.stream(

            fn=
                live_predict,

            inputs=
                webcam,

            outputs=[

                visual_output,

                recognized_text

            ],

            time_limit=
                120,

            stream_every=
                0.15

        )


        clear_button.click(

            fn=
                clear_live_sequence,

            outputs=
                recognized_text

        )


    demo.launch(

        share=True,

        debug=False

    )

In [ ]:
# ======================================================================
# 🤟 SIGNBRIDGE AI — LIVE DEMO ONLY
# ======================================================================
# PURPOSE:
#   • Does NOT download FSBoard
#   • Does NOT add training samples
#   • Does NOT retrain the model
#   • Loads the existing best model from Google Drive
#   • Starts MediaPipe hand tracking
#   • Launches a fresh Gradio webcam link
#
# Use this cell whenever the previous Gradio link expires.
# ======================================================================


# ======================================================================
# 0. INSTALL ONLY THE DEMO REQUIREMENTS
# ======================================================================

!pip -q install mediapipe gradio opencv-python-headless


# ======================================================================
# 1. IMPORTS
# ======================================================================

import os
import json
import urllib.request
from pathlib import Path
from collections import deque

import numpy as np
import cv2
import tensorflow as tf
import mediapipe as mp
import gradio as gr

print("=" * 80)
print("🤟 SIGNBRIDGE AI — LIVE DEMO ONLY")
print("=" * 80)


# ======================================================================
# 2. MOUNT GOOGLE DRIVE
# ======================================================================

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("✅ Google Drive already mounted.")


# ======================================================================
# 3. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SignBridge_FSBoard"
)

MODEL_DIR = (
    PROJECT_DIR /
    "model_v3"
)

MODEL_PATH = (
    MODEL_DIR /
    "best_signbridge.keras"
)

PROJECT_INFO_PATH = (
    MODEL_DIR /
    "project_info.json"
)

HAND_MODEL_PATH = Path(
    "/content/hand_landmarker.task"
)


# ======================================================================
# 4. CHECK SAVED MODEL
# ======================================================================

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"""
❌ SignBridge trained model was not found.

Expected location:

{MODEL_PATH}

Run the MASTER training cell first.
"""
    )

print(
    "✅ Saved model found:",
    MODEL_PATH
)


# ======================================================================
# 5. LOAD VOCABULARY
# ======================================================================

# Exact vocabulary used by the working SignBridge model.
# First model output class is reserved for CTC blank (index 0).

DEFAULT_CHARACTERS = (
    " !#%&'()*+,-./0123456789:=?@[_abcdefghijklmnopqrstuvwxyz~"
)

CHARACTERS = DEFAULT_CHARACTERS


# Prefer vocabulary saved by the master pipeline when available.

if PROJECT_INFO_PATH.exists():

    try:

        with open(
            PROJECT_INFO_PATH,
            "r"
        ) as f:

            project_info = json.load(f)

        saved_characters = project_info.get(
            "characters"
        )

        if saved_characters:

            if isinstance(
                saved_characters,
                list
            ):

                CHARACTERS = "".join(
                    saved_characters
                )

            else:

                CHARACTERS = str(
                    saved_characters
                )

            print(
                "✅ Vocabulary loaded from project_info.json."
            )

    except Exception as e:

        print(
            "⚠️ Could not read saved project vocabulary."
        )

        print(
            "Using known SignBridge vocabulary."
        )


CHAR_TO_ID = {
    character: index + 1
    for index, character
    in enumerate(CHARACTERS)
}

ID_TO_CHAR = {
    index + 1: character
    for index, character
    in enumerate(CHARACTERS)
}

NUM_CLASSES = (
    len(CHARACTERS)
    +
    1
)

print(
    "Character classes:",
    len(CHARACTERS)
)

print(
    "Classes including CTC blank:",
    NUM_CLASSES
)


# ======================================================================
# 6. LOAD TRAINED SIGNBRIDGE MODEL
# ======================================================================

print(
    "\nLoading trained SignBridge model..."
)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print(
    "✅ Best SignBridge model loaded."
)

print(
    "Model input :",
    model.input_shape
)

print(
    "Model output:",
    model.output_shape
)


# Safety check

if model.output_shape[-1] != NUM_CLASSES:

    raise RuntimeError(
        f"""
❌ Vocabulary/model mismatch.

Model classes:
{model.output_shape[-1]}

Vocabulary classes:
{NUM_CLASSES}

Do not retrain anything.
Share this message with me if it appears.
"""
    )


# ======================================================================
# 7. MEDIAPIPE HAND LANDMARKER MODEL
# ======================================================================

HAND_MODEL_URL = (
    "https://storage.googleapis.com/"
    "mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/"
    "hand_landmarker.task"
)


if not HAND_MODEL_PATH.exists():

    print(
        "\nDownloading MediaPipe Hand Landmarker..."
    )

    urllib.request.urlretrieve(
        HAND_MODEL_URL,
        HAND_MODEL_PATH
    )


print(
    "✅ MediaPipe Hand Landmarker model ready."
)


# ======================================================================
# 8. CREATE MEDIAPIPE TASKS LANDMARKER
# ======================================================================

BaseOptions = mp.tasks.BaseOptions

HandLandmarker = (
    mp.tasks.vision.HandLandmarker
)

HandLandmarkerOptions = (
    mp.tasks.vision.HandLandmarkerOptions
)

RunningMode = (
    mp.tasks.vision.RunningMode
)


live_options = HandLandmarkerOptions(

    base_options=BaseOptions(
        model_asset_path=str(
            HAND_MODEL_PATH
        )
    ),

    running_mode=RunningMode.IMAGE,

    num_hands=1,

    min_hand_detection_confidence=0.50,

    min_hand_presence_confidence=0.50,

    min_tracking_confidence=0.50
)


live_landmarker = (
    HandLandmarker.create_from_options(
        live_options
    )
)

print(
    "✅ MediaPipe Tasks hand tracking ready."
)


# ======================================================================
# 9. LANDMARK NORMALIZATION
# ======================================================================

NUM_LANDMARKS = 21

FEATURES = 63


def normalize_landmarks(
    xyz
):

    """
    Same normalization principle used during training.

    1. Wrist becomes origin.
    2. Scale using Wrist → Middle Finger MCP distance.
    """

    xyz = np.asarray(
        xyz,
        dtype=np.float32
    ).reshape(
        NUM_LANDMARKS,
        3
    )


    # Wrist = landmark 0

    wrist = xyz[
        0:1
    ]

    centered = (
        xyz
        -
        wrist
    )


    # Middle Finger MCP = landmark 9

    scale = np.linalg.norm(
        centered[
            9
        ]
    )


    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):

        scale = 1.0


    normalized = (
        centered
        /
        scale
    )


    normalized = np.nan_to_num(
        normalized,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


    return normalized.astype(
        np.float32
    )


# ======================================================================
# 10. CTC DECODER
# ======================================================================

def decode_ids(
    ids
):

    characters = []

    for idx in ids:

        idx = int(
            idx
        )

        if idx == 0:
            continue

        character = ID_TO_CHAR.get(
            idx,
            ""
        )

        characters.append(
            character
        )

    return "".join(
        characters
    )


def decode_predictions(
    logits,
    lengths=None
):

    """
    Greedy CTC decoder.

    Performs:
      argmax
      → collapse repeated IDs
      → remove CTC blank
      → characters
    """

    if tf.is_tensor(
        logits
    ):

        logits = logits.numpy()


    pred_ids = np.argmax(
        logits,
        axis=-1
    )


    if lengths is None:

        lengths = [
            pred_ids.shape[1]
        ] * pred_ids.shape[0]


    elif tf.is_tensor(
        lengths
    ):

        lengths = lengths.numpy()


    outputs = []


    for row, length in zip(
        pred_ids,
        lengths
    ):

        row = row[
            :int(length)
        ]

        collapsed = []

        previous = None


        for idx in row:

            idx = int(
                idx
            )


            # CTC collapse repeated predictions

            if idx != previous:

                if idx != 0:

                    collapsed.append(
                        idx
                    )

            previous = idx


        outputs.append(
            decode_ids(
                collapsed
            )
        )


    return outputs


# ======================================================================
# 11. HAND SKELETON CONNECTIONS
# ======================================================================

HAND_CONNECTIONS = [

    # Thumb

    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),

    # Index

    (0, 5),
    (5, 6),
    (6, 7),
    (7, 8),

    # Middle

    (5, 9),
    (9, 10),
    (10, 11),
    (11, 12),

    # Ring

    (9, 13),
    (13, 14),
    (14, 15),
    (15, 16),

    # Pinky

    (13, 17),
    (17, 18),
    (18, 19),
    (19, 20),

    # Palm

    (0, 17)

]


# ======================================================================
# 12. LIVE SEQUENCE BUFFER
# ======================================================================

LIVE_WINDOW = 100

MIN_LIVE_FRAMES = 12


LIVE_BUFFER = deque(
    maxlen=LIVE_WINDOW
)


# ======================================================================
# 13. LIVE PREDICTION
# ======================================================================

def live_predict(
    frame
):

    if frame is None:

        return (
            None,
            "Waiting for camera..."
        )


    image = np.asarray(
        frame
    ).copy()


    # --------------------------------------------------------------
    # MediaPipe requires RGB SRGB image
    # --------------------------------------------------------------

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(
            image.astype(
                np.uint8
            )
        )
    )


    result = (
        live_landmarker.detect(
            mp_image
        )
    )


    hand_detected = False


    # --------------------------------------------------------------
    # HAND FOUND
    # --------------------------------------------------------------

    if result.hand_landmarks:

        hand_detected = True

        hand = result.hand_landmarks[
            0
        ]


        xyz = np.asarray(

            [
                [
                    landmark.x,
                    landmark.y,
                    landmark.z
                ]

                for landmark
                in hand
            ],

            dtype=np.float32
        )


        xyz = normalize_landmarks(
            xyz
        )


        LIVE_BUFFER.append(
            xyz.reshape(
                FEATURES
            )
        )


        # ----------------------------------------------------------
        # DRAW HAND LANDMARKS
        # ----------------------------------------------------------

        height, width = image.shape[
            :2
        ]


        points = []


        for landmark in hand:

            px = int(
                landmark.x
                *
                width
            )

            py = int(
                landmark.y
                *
                height
            )


            points.append(
                (
                    px,
                    py
                )
            )


        for start_idx, end_idx in HAND_CONNECTIONS:

            if (
                start_idx < len(points)
                and
                end_idx < len(points)
            ):

                cv2.line(

                    image,

                    points[
                        start_idx
                    ],

                    points[
                        end_idx
                    ],

                    (
                        0,
                        255,
                        0
                    ),

                    2,

                    cv2.LINE_AA
                )


        for px, py in points:

            cv2.circle(

                image,

                (
                    px,
                    py
                ),

                4,

                (
                    0,
                    255,
                    0
                ),

                -1
            )


    # --------------------------------------------------------------
    # RECOGNITION
    # --------------------------------------------------------------

    if len(
        LIVE_BUFFER
    ) >= MIN_LIVE_FRAMES:

        sequence = np.asarray(

            LIVE_BUFFER,

            dtype=np.float32

        )[

            None,
            ...

        ]


        logits = model(

            sequence,

            training=False
        )


        prediction = decode_predictions(
            logits
        )[0]


        if not prediction:

            prediction = (
                "Recognizing..."
            )


    else:

        prediction = (
            f"Collecting movement "
            f"{len(LIVE_BUFFER)}/"
            f"{MIN_LIVE_FRAMES}"
        )


    # --------------------------------------------------------------
    # STATUS OVERLAY
    # --------------------------------------------------------------

    overlay_height = 100


    cv2.rectangle(

        image,

        (
            0,
            0
        ),

        (
            image.shape[1],
            overlay_height
        ),

        (
            0,
            0,
            0
        ),

        -1
    )


    status = (
        "Hand detected"
        if hand_detected
        else
        "Show signing hand"
    )


    cv2.putText(

        image,

        status,

        (
            15,
            30
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.65,

        (
            255,
            255,
            255
        ),

        2,

        cv2.LINE_AA
    )


    display_prediction = (
        prediction[
            :55
        ]
    )


    cv2.putText(

        image,

        "SignBridge: "
        +
        display_prediction,

        (
            15,
            72
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.65,

        (
            255,
            255,
            255
        ),

        2,

        cv2.LINE_AA
    )


    return (
        image,
        prediction
    )


# ======================================================================
# 14. CLEAR / NEW PHRASE
# ======================================================================

def clear_live():

    LIVE_BUFFER.clear()

    return (
        "Sequence cleared. "
        "Start fingerspelling."
    )


# ======================================================================
# 15. GRADIO LIVE DEMO
# ======================================================================

print(
    "\n" +
    "=" * 80
)

print(
    "STARTING SIGNBRIDGE LIVE DEMO"
)

print(
    "=" * 80
)


with gr.Blocks(
    title="SignBridge AI"
) as demo:


    gr.Markdown(
        """
# 🤟 SignBridge AI

## Real-Time ASL Fingerspelling → Text

**MediaPipe → Temporal CNN → BiGRU → CTC**

### How to test

1. Click **Clear / Start New Phrase**
2. Keep your signing hand clearly visible.
3. Fingerspell naturally.
4. Watch the recognized text update.
5. Press **Clear / Start New Phrase** before testing another word or phrase.

The model analyses a sequence of hand movements rather than a single image.
"""
    )


    with gr.Row():

        webcam = gr.Image(

            sources=[
                "webcam"
            ],

            type="numpy",

            streaming=True,

            label="Live Camera"
        )


        visual_output = gr.Image(

            label=
            "Hand Tracking + Recognition"
        )


    recognized_text = gr.Textbox(

        label=
        "Recognized Fingerspelling",

        lines=2
    )


    clear_button = gr.Button(
        "Clear / Start New Phrase"
    )


    webcam.stream(

        fn=live_predict,

        inputs=webcam,

        outputs=[
            visual_output,
            recognized_text
        ]
    )


    clear_button.click(

        fn=clear_live,

        outputs=recognized_text
    )


# ======================================================================
# 16. LAUNCH
# ======================================================================

demo.launch(
    share=True,
    debug=False
)

In [ ]:
# ======================================================================
# 🤟 SIGNBRIDGE AI — UNSEEN FSBOARD FUNCTIONALITY TEST
# ======================================================================
#
# This is a completely separate diagnostic cell.
#
# DOES:
#   ✓ Load the existing trained model
#   ✓ Pick a RANDOM sample from the official FSBoard TEST split
#   ✓ Prefer a simple single alphabetic word for easy visual checking
#   ✓ Download ONE test video only
#   ✓ Extract MediaPipe landmarks
#   ✓ Run SignBridge prediction
#   ✓ Print Ground Truth / Prediction / CER
#   ✓ Display the actual FSBoard video
#   ✓ Plot character probabilities through time using Matplotlib
#   ✓ Delete temporary raw video afterwards
#
# DOES NOT:
#   ✗ Train the model
#   ✗ Modify the model
#   ✗ Add this file to training
#   ✗ Modify your Master Cell
#
# ======================================================================


# ======================================================================
# 0. INSTALL REQUIRED PACKAGES
# ======================================================================

!pip -q install "pandas==2.2.3" kagglehub kaggle mediapipe opencv-python-headless matplotlib


# ======================================================================
# 1. IMPORTS
# ======================================================================

import os
import json
import random
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
import mediapipe as mp
import matplotlib.pyplot as plt
import kagglehub

from IPython.display import display, Video


print("=" * 80)
print("🤟 SIGNBRIDGE AI — UNSEEN TEST SAMPLE CHECK")
print("=" * 80)


# ======================================================================
# 2. GOOGLE DRIVE
# ======================================================================

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("✅ Google Drive already mounted.")


# ======================================================================
# 3. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SignBridge_FSBoard"
)

MODEL_DIR = (
    PROJECT_DIR /
    "model_v3"
)

MODEL_PATH = (
    MODEL_DIR /
    "best_signbridge.keras"
)

PROJECT_INFO_PATH = (
    MODEL_DIR /
    "project_info.json"
)

TEST_METADATA_PATH = (
    PROJECT_DIR /
    "metadata/daun_v3/metadata/daun_v3-test.json"
)

TEMP_DIR = Path(
    "/content/signbridge_unseen_test"
)

HAND_MODEL_PATH = Path(
    "/content/hand_landmarker.task"
)

DATASET = "googleai/fsboard"


# ======================================================================
# 4. CHECK FILES
# ======================================================================

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"""
❌ Saved SignBridge model not found:

{MODEL_PATH}
"""
    )


if not TEST_METADATA_PATH.exists():

    raise FileNotFoundError(
        f"""
❌ FSBoard test metadata not found:

{TEST_METADATA_PATH}
"""
    )


print("✅ Model found.")
print("✅ Official FSBoard TEST metadata found.")


# ======================================================================
# 5. KAGGLE AUTHENTICATION
# ======================================================================

try:

    from google.colab import userdata

    token = userdata.get(
        "KAGGLE_API_TOKEN"
    )

    if token:

        os.environ[
            "KAGGLE_API_TOKEN"
        ] = token

        print(
            "✅ Kaggle authentication configured."
        )

except Exception:

    print(
        "ℹ️ Using existing Kaggle authentication."
    )


# ======================================================================
# 6. LOAD SAVED PROJECT VOCABULARY
# ======================================================================

DEFAULT_CHARACTERS = (
    " !#%&'()*+,-./0123456789:=?@[_abcdefghijklmnopqrstuvwxyz~"
)

CHARACTERS = DEFAULT_CHARACTERS


if PROJECT_INFO_PATH.exists():

    try:

        with open(
            PROJECT_INFO_PATH,
            "r",
            encoding="utf-8"
        ) as f:

            project_info = json.load(f)


        saved_characters = (
            project_info.get(
                "characters"
            )
        )


        if saved_characters:

            if isinstance(
                saved_characters,
                list
            ):

                CHARACTERS = "".join(
                    saved_characters
                )

            else:

                CHARACTERS = str(
                    saved_characters
                )


            print(
                "✅ Vocabulary loaded from saved project."
            )

    except Exception as e:

        print(
            "⚠️ Could not load project_info.json."
        )

        print(
            "Using known working vocabulary."
        )


CHAR_TO_ID = {

    character:
        index + 1

    for index, character
    in enumerate(
        CHARACTERS
    )
}


ID_TO_CHAR = {

    index + 1:
        character

    for index, character
    in enumerate(
        CHARACTERS
    )
}


NUM_CLASSES = (
    len(
        CHARACTERS
    )
    +
    1
)


print(
    "Vocabulary characters:",
    repr(
        CHARACTERS
    )
)

print(
    "Output classes:",
    NUM_CLASSES
)


# ======================================================================
# 7. LOAD TRAINED MODEL
# ======================================================================

print(
    "\nLoading SignBridge model..."
)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print(
    "✅ Best trained model loaded."
)

print(
    "Input shape :",
    model.input_shape
)

print(
    "Output shape:",
    model.output_shape
)


if model.output_shape[-1] != NUM_CLASSES:

    raise RuntimeError(
        f"""
❌ Model/vocabulary mismatch.

Model has:
{model.output_shape[-1]} classes

Vocabulary expects:
{NUM_CLASSES}

Share this message with me if you see it.
"""
    )


# ======================================================================
# 8. LOAD OFFICIAL TEST METADATA
# ======================================================================

with open(
    TEST_METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    raw_test_records = json.load(
        f
    )


test_records = []


for item in raw_test_records:

    # FSBoard fingerspelling only

    if item.get(
        "signingType"
    ) != "fs":

        continue


    clip = item.get(
        "clipFilename"
    )

    phrase = item.get(
        "phrase"
    )


    if not clip or phrase is None:
        continue


    phrase = str(
        phrase
    ).strip().lower()


    # --------------------------------------------------------------
    # For this visual diagnostic we intentionally prefer:
    #
    #     one simple alphabetic word
    #
    # rather than URLs / addresses / numbers.
    #
    # This does NOT change the model.
    # It simply makes the test easier to understand visually.
    # --------------------------------------------------------------

    if not phrase.isalpha():
        continue


    if not (
        4
        <=
        len(phrase)
        <=
        12
    ):
        continue


    remote_path = (

        "daun_v3/video_clips/"
        "daun_v3-test/"
        +
        clip
    )


    test_records.append(

        {
            "filename":
                clip,

            "phrase":
                phrase,

            "signer":
                str(
                    item.get(
                        "signerId",
                        "unknown"
                    )
                ),

            "remote_path":
                remote_path
        }

    )


print(
    "\nEligible simple unseen test words:",
    f"{len(test_records):,}"
)


if not test_records:

    raise RuntimeError(
        "No suitable simple test words found."
    )


# ======================================================================
# 9. SELECT RANDOM UNSEEN TEST SAMPLE
# ======================================================================

sample = random.choice(
    test_records
)


GROUND_TRUTH = sample[
    "phrase"
]

REMOTE_PATH = sample[
    "remote_path"
]


print(
    "\n" +
    "=" * 80
)

print(
    "SELECTED OFFICIAL FSBOARD TEST SAMPLE"
)

print(
    "=" * 80
)

print(
    "Ground truth :",
    repr(
        GROUND_TRUTH
    )
)

print(
    "Signer ID    :",
    sample[
        "signer"
    ]
)

print(
    "Filename     :",
    sample[
        "filename"
    ]
)

print()
print(
    "✅ This clip comes from the official TEST split."
)

print(
    "✅ It is NOT used for gradient-based model training."
)


# ======================================================================
# 10. DOWNLOAD ONE TEST VIDEO ONLY
# ======================================================================

shutil.rmtree(
    TEMP_DIR,
    ignore_errors=True
)

TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "\nDownloading ONE temporary test clip..."
)


download_result = kagglehub.dataset_download(

    DATASET,

    path=REMOTE_PATH,

    output_dir=str(
        TEMP_DIR
    )
)


download_result = Path(
    download_result
)


# KaggleHub may return either the file itself
# or a containing directory.

if download_result.is_file():

    VIDEO_PATH = download_result

else:

    possible_files = list(
        TEMP_DIR.rglob(
            sample[
                "filename"
            ]
        )
    )

    if not possible_files:

        raise FileNotFoundError(
            "Downloaded video could not be located."
        )

    VIDEO_PATH = possible_files[
        0
    ]


print(
    "✅ Test video downloaded:"
)

print(
    VIDEO_PATH
)


# ======================================================================
# 11. DISPLAY ORIGINAL TEST VIDEO
# ======================================================================

print(
    "\n" +
    "=" * 80
)

print(
    "ORIGINAL FSBOARD TEST VIDEO"
)

print(
    "=" * 80
)

print(
    "Watch the signer carefully."
)

print(
    "Expected word:",
    GROUND_TRUTH.upper()
)


display(

    Video(

        str(
            VIDEO_PATH
        ),

        embed=True,

        width=500
    )

)


# ======================================================================
# 12. MEDIAPIPE HAND LANDMARKER
# ======================================================================

HAND_MODEL_URL = (
    "https://storage.googleapis.com/"
    "mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/"
    "hand_landmarker.task"
)


if not HAND_MODEL_PATH.exists():

    print(
        "\nDownloading MediaPipe Hand Landmarker..."
    )

    urllib.request.urlretrieve(
        HAND_MODEL_URL,
        HAND_MODEL_PATH
    )


BaseOptions = (
    mp.tasks.BaseOptions
)

HandLandmarker = (
    mp.tasks.vision.HandLandmarker
)

HandLandmarkerOptions = (
    mp.tasks.vision.HandLandmarkerOptions
)

RunningMode = (
    mp.tasks.vision.RunningMode
)


video_options = HandLandmarkerOptions(

    base_options=BaseOptions(
        model_asset_path=str(
            HAND_MODEL_PATH
        )
    ),

    running_mode=RunningMode.VIDEO,

    num_hands=1,

    min_hand_detection_confidence=0.35,

    min_hand_presence_confidence=0.35,

    min_tracking_confidence=0.35
)


print(
    "✅ MediaPipe VIDEO-mode detector ready."
)


# ======================================================================
# 13. PREPROCESSING SETTINGS — SAME DESIGN AS MASTER PIPELINE
# ======================================================================

TARGET_FPS = 12

MAX_FRAMES = 240

MIN_FRAMES = 6

NUM_LANDMARKS = 21

FEATURES = 63


# ======================================================================
# 14. NORMALIZE LANDMARKS
# ======================================================================

def normalize_landmarks(
    xyz
):

    xyz = np.asarray(
        xyz,
        dtype=np.float32
    )


    # Wrist becomes origin

    xyz = (
        xyz
        -
        xyz[
            0
        ]
    )


    # Scale using wrist → middle-finger MCP

    scale = np.linalg.norm(
        xyz[
            9
        ]
    )


    if (
        not np.isfinite(
            scale
        )
        or
        scale < 1e-6
    ):

        scale = 1.0


    xyz = (
        xyz
        /
        scale
    )


    xyz = np.nan_to_num(

        xyz,

        nan=0.0,

        posinf=0.0,

        neginf=0.0
    )


    return xyz.astype(
        np.float32
    )


# ======================================================================
# 15. VIDEO → LANDMARK SEQUENCE
# ======================================================================

def extract_landmarks_from_video(
    video_path
):

    capture = cv2.VideoCapture(
        str(
            video_path
        )
    )


    if not capture.isOpened():

        return None


    source_fps = capture.get(
        cv2.CAP_PROP_FPS
    )


    if (
        not source_fps
        or
        np.isnan(
            source_fps
        )
        or
        source_fps <= 0
    ):

        source_fps = 30.0


    frame_step = max(

        1,

        int(
            round(
                source_fps
                /
                TARGET_FPS
            )
        )
    )


    sequence = []

    frame_number = 0

    timestamps = []


    with HandLandmarker.create_from_options(
        video_options
    ) as detector:


        while True:

            success, frame = (
                capture.read()
            )


            if not success:
                break


            use_frame = (

                frame_number
                %
                frame_step
                ==
                0
            )


            current_frame_number = (
                frame_number
            )

            frame_number += 1


            if not use_frame:
                continue


            rgb = cv2.cvtColor(

                frame,

                cv2.COLOR_BGR2RGB
            )


            mp_image = mp.Image(

                image_format=
                    mp.ImageFormat.SRGB,

                data=
                    np.ascontiguousarray(
                        rgb
                    )
            )


            timestamp_ms = int(

                (
                    current_frame_number
                    /
                    source_fps
                )
                *
                1000
            )


            result = (
                detector.detect_for_video(

                    mp_image,

                    timestamp_ms
                )
            )


            if result.hand_landmarks:

                hand = (
                    result.hand_landmarks[
                        0
                    ]
                )


                xyz = np.asarray(

                    [

                        [
                            lm.x,
                            lm.y,
                            lm.z
                        ]

                        for lm
                        in hand

                    ],

                    dtype=np.float32
                )


                xyz = (
                    normalize_landmarks(
                        xyz
                    )
                )


                sequence.append(

                    xyz.reshape(
                        FEATURES
                    )
                )


                timestamps.append(
                    timestamp_ms
                    /
                    1000.0
                )


            elif sequence:

                # Preserve timing during a short
                # MediaPipe detection dropout.

                sequence.append(

                    sequence[
                        -1
                    ].copy()

                )


                timestamps.append(
                    timestamp_ms
                    /
                    1000.0
                )


            if (
                len(
                    sequence
                )
                >=
                MAX_FRAMES
            ):

                break


    capture.release()


    if (
        len(
            sequence
        )
        <
        MIN_FRAMES
    ):

        return None


    return (

        np.asarray(
            sequence,
            dtype=np.float32
        ),

        np.asarray(
            timestamps,
            dtype=np.float32
        )

    )


# ======================================================================
# 16. EXTRACT TEST FEATURES
# ======================================================================

print(
    "\nExtracting MediaPipe hand landmarks..."
)


extracted = (
    extract_landmarks_from_video(
        VIDEO_PATH
    )
)


if extracted is None:

    raise RuntimeError(
        """
❌ MediaPipe could not obtain enough hand frames
from this randomly selected clip.

Run this cell again to select another test sample.
"""
    )


landmarks, time_seconds = (
    extracted
)


print(
    "✅ Frames extracted:",
    len(
        landmarks
    )
)

print(
    "✅ Feature vector:",
    landmarks.shape
)


# ======================================================================
# 17. CTC DECODER
# ======================================================================

def decode_ids(
    ids
):

    output = []


    for value in ids:

        value = int(
            value
        )


        if value <= 0:
            continue


        character = (
            ID_TO_CHAR.get(
                value,
                ""
            )
        )


        output.append(
            character
        )


    return "".join(
        output
    )


def decode_predictions(
    logits
):

    if tf.is_tensor(
        logits
    ):

        logits = logits.numpy()


    predicted_ids = np.argmax(
        logits,
        axis=-1
    )


    outputs = []


    for row in predicted_ids:

        collapsed = []

        previous = None


        for value in row:

            value = int(
                value
            )


            # Standard CTC collapsing

            if value != previous:

                if value != 0:

                    collapsed.append(
                        value
                    )


            previous = value


        outputs.append(
            decode_ids(
                collapsed
            )
        )


    return outputs


# ======================================================================
# 18. RUN MODEL PREDICTION
# ======================================================================

input_sequence = landmarks[
    None,
    ...
]


print(
    "\nRunning SignBridge inference..."
)


logits = model(

    input_sequence,

    training=False

)


PREDICTION = (
    decode_predictions(
        logits
    )[0]
)


print(
    "✅ Prediction complete."
)


# ======================================================================
# 19. CHARACTER ERROR RATE
# ======================================================================

def levenshtein_distance(
    reference,
    hypothesis
):

    rows = (
        len(
            reference
        )
        +
        1
    )

    cols = (
        len(
            hypothesis
        )
        +
        1
    )


    matrix = np.zeros(

        (
            rows,
            cols
        ),

        dtype=np.int32
    )


    matrix[
        :,
        0
    ] = np.arange(
        rows
    )


    matrix[
        0,
        :
    ] = np.arange(
        cols
    )


    for i in range(
        1,
        rows
    ):

        for j in range(
            1,
            cols
        ):

            substitution_cost = (

                0

                if
                reference[
                    i - 1
                ]
                ==
                hypothesis[
                    j - 1
                ]

                else
                1
            )


            matrix[
                i,
                j
            ] = min(

                matrix[
                    i - 1,
                    j
                ]
                +
                1,

                matrix[
                    i,
                    j - 1
                ]
                +
                1,

                matrix[
                    i - 1,
                    j - 1
                ]
                +
                substitution_cost
            )


    return int(
        matrix[
            -1,
            -1
        ]
    )


edit_distance = (
    levenshtein_distance(

        GROUND_TRUTH,

        PREDICTION

    )
)


CER = (

    edit_distance
    /
    max(
        1,
        len(
            GROUND_TRUTH
        )
    )

)


print(
    "\n" +
    "=" * 80
)

print(
    "SIGNBRIDGE FUNCTIONALITY RESULT"
)

print(
    "=" * 80
)

print(
    f"GROUND TRUTH : {GROUND_TRUTH}"
)

print(
    f"PREDICTION   : {PREDICTION}"
)

print(
    f"EDIT ERRORS  : {edit_distance}"
)

print(
    f"CER          : {CER * 100:.2f}%"
)


if PREDICTION == GROUND_TRUTH:

    print(
        "\n🎉 EXACT MATCH — functionality verified."
    )

elif CER <= 0.20:

    print(
        "\n✅ VERY CLOSE prediction."
    )

elif CER <= 0.40:

    print(
        "\n🟡 Recognizable but contains character errors."
    )

else:

    print(
        "\n🔴 Difficult sample for the current model."
    )


# ======================================================================
# 20. CONVERT LOGITS → PROBABILITIES
# ======================================================================

probabilities = tf.nn.softmax(

    logits,

    axis=-1

).numpy()[0]


# ======================================================================
# 21. MATPLOTLIB — CHARACTER PROBABILITY HEATMAP
# ======================================================================

# We plot:
#
# Ground-truth letters +
# predicted letters
#
# rather than all 57 characters,
# so the graph remains readable.

interesting_characters = []


for character in (
    GROUND_TRUTH
    +
    PREDICTION
):

    if (
        character
        in CHAR_TO_ID
        and
        character
        not in interesting_characters
    ):

        interesting_characters.append(
            character
        )


# If too few distinct characters were found,
# add the model's strongest non-blank classes.

mean_probabilities = (
    probabilities.mean(
        axis=0
    )
)


top_ids = np.argsort(
    mean_probabilities[
        1:
    ]
)[
    ::-1
][
    :10
] + 1


for class_id in top_ids:

    character = (
        ID_TO_CHAR.get(
            int(
                class_id
            ),
            ""
        )
    )

    if (
        character
        and
        character
        not in interesting_characters
    ):

        interesting_characters.append(
            character
        )


    if (
        len(
            interesting_characters
        )
        >=
        12
    ):

        break


character_ids = [

    CHAR_TO_ID[
        character
    ]

    for character
    in interesting_characters

]


heatmap_data = (

    probabilities[
        :,
        character_ids
    ].T

)


# Use model time steps if timestamps do not
# exactly equal the output sequence length.

if (
    len(
        time_seconds
    )
    ==
    probabilities.shape[
        0
    ]
):

    x_axis = time_seconds

    x_label = (
        "Video time (seconds)"
    )

else:

    x_axis = np.arange(
        probabilities.shape[
            0
        ]
    )

    x_label = (
        "Model time step"
    )


plt.figure(
    figsize=(
        14,
        6
    )
)


image = plt.imshow(

    heatmap_data,

    aspect="auto",

    origin="lower",

    interpolation="nearest",

    extent=[

        float(
            x_axis[
                0
            ]
        ),

        float(
            x_axis[
                -1
            ]
        ),

        -0.5,

        len(
            interesting_characters
        )
        -
        0.5

    ]

)


plt.colorbar(
    image,
    label="Model probability"
)


plt.yticks(

    range(
        len(
            interesting_characters
        )
    ),

    [
        repr(
            character
        )

        for character
        in interesting_characters
    ]

)


plt.xlabel(
    x_label
)


plt.ylabel(
    "Character"
)


plt.title(

    "SignBridge AI — Character Probability Through Time\n"
    +
    f"Ground Truth: {GROUND_TRUTH.upper()}   |   "
    +
    f"Prediction: {PREDICTION.upper()}   |   "
    +
    f"CER: {CER * 100:.2f}%"

)


plt.tight_layout()

plt.show()


# ======================================================================
# 22. SECOND MATPLOTLIB GRAPH — FRAME-WISE BEST CHARACTER
# ======================================================================

best_class_per_frame = np.argmax(

    probabilities,

    axis=-1

)


best_confidence = np.max(

    probabilities,

    axis=-1

)


frame_labels = []


for class_id in best_class_per_frame:

    class_id = int(
        class_id
    )


    if class_id == 0:

        frame_labels.append(
            "<blank>"
        )

    else:

        frame_labels.append(

            ID_TO_CHAR.get(
                class_id,
                "?"
            )

        )


# Convert labels to numeric positions
# for a readable categorical timeline.

unique_frame_labels = []


for label in frame_labels:

    if (
        label
        not in unique_frame_labels
    ):

        unique_frame_labels.append(
            label
        )


label_to_y = {

    label:
        index

    for index, label
    in enumerate(
        unique_frame_labels
    )

}


frame_y = [

    label_to_y[
        label
    ]

    for label
    in frame_labels

]


plt.figure(
    figsize=(
        14,
        5
    )
)


plt.scatter(

    x_axis,

    frame_y,

    s=20
)


plt.plot(

    x_axis,

    frame_y,

    alpha=0.35
)


plt.yticks(

    range(
        len(
            unique_frame_labels
        )
    ),

    unique_frame_labels

)


plt.xlabel(
    x_label
)


plt.ylabel(
    "Highest-probability CTC class"
)


plt.title(

    "SignBridge AI — Raw CTC Character Alignment\n"
    +
    f"Target: {GROUND_TRUTH.upper()}   →   "
    +
    f"Decoded: {PREDICTION.upper()}"

)


plt.grid(
    alpha=0.25
)


plt.tight_layout()

plt.show()


# ======================================================================
# 23. TEXT SUMMARY FOR PROJECT SCREENSHOT
# ======================================================================

print(
    "\n" +
    "=" * 80
)

print(
    "FUNCTIONALITY CHECK SUMMARY"
)

print(
    "=" * 80
)

print(
    f"""
Dataset Split      : Official FSBoard TEST
Training Usage     : NOT used for model gradient training
Signer             : {sample['signer']}
Video              : {sample['filename']}

Expected Text       : {GROUND_TRUTH}
Predicted Text      : {PREDICTION}

Character Errors   : {edit_distance}
Character Error Rate: {CER * 100:.2f}%

Input Landmark Shape:
{landmarks.shape}

Architecture:
MediaPipe Hand Landmarks
→ 63 features/frame
→ Temporal Conv1D CNN
→ Residual CNN
→ Bidirectional GRU
→ Bidirectional GRU
→ CTC
→ Text
"""
)


# ======================================================================
# 24. CLEAN TEMPORARY RAW VIDEO
# ======================================================================

shutil.rmtree(
    TEMP_DIR,
    ignore_errors=True
)


print(
    "✅ Temporary raw test video deleted."
)

print(
    "✅ Saved training data and model were NOT modified."
)

print(
    "\nRun this cell again to test a DIFFERENT random unseen test word."
)